In [2]:
import cv2
import numpy as np
import requests
import time

API_URL = "http://127.0.0.1:8000/predict"

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

if not cap.isOpened():
    print("Error: Can't receive frame.")
    exit()

font = cv2.FONT_HERSHEY_SIMPLEX

print("Клиент запущен. Подключение к API...")

while True:
    start_time = time.time()
    ret, frame = cap.read()
    if not ret:
        break

    display_frame = frame.copy()
    
    _, img_encoded = cv2.imencode('.jpg', frame)
    
    try:
        files = {"file": bytes(img_encoded)}
        response = requests.post(API_URL, files=files)
        
        if response.status_code == 200:
            data = response.json()
            
            if data.get("object_found"):
                x1, y1, x2, y2 = data["box"]
                class_name = data["class_name"]
                conf = data["confidence"]

                cv2.rectangle(display_frame, (x1, y1), (x2, y2), (0, 255, 0), 3)
                label = f"{class_name}: {conf*100:.1f}%"
                (tw, th), _ = cv2.getTextSize(label, font, 0.8, 2)
                cv2.rectangle(display_frame, (x1, y1 - 30), (x1 + tw, y1), (0, 255, 0), -1)
                cv2.putText(display_frame, label, (x1, y1 - 5), font, 0.8, (0, 0, 0), 2)
                
                try:
                    object_crop = frame[y1:y2, x1:x2]
                    if object_crop.size > 0:
                        debug_crop = cv2.resize(object_crop, (150, 150))
                        display_frame[10:160, 10:160] = debug_crop
                        cv2.rectangle(display_frame, (10, 10), (160, 160), (255, 255, 255), 1)
                except Exception:
                    pass
            else:
                cv2.putText(display_frame, "Searching for animal...", (20, 50), font, 1, (0, 0, 255), 2)
        else:
            cv2.putText(display_frame, f"API Error: {response.status_code}", (20, 50), font, 1, (0, 0, 255), 2)
            
    except requests.exceptions.ConnectionError:
        cv2.putText(display_frame, "API disconnected. Check server!", (20, 50), font, 1, (0, 0, 255), 2)

    fps = 1.0 / (time.time() - start_time)
    cv2.putText(display_frame, f"FPS: {fps:.1f}", (display_frame.shape[1] - 150, 40), font, 0.7, (200, 200, 200), 2)

    cv2.imshow("Smart Animal Classifier (Client)", display_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Клиент запущен. Подключение к API...
